# Complete Trading Environment Implementation

In [5]:
# trading_env.py
"""
Complete Trading Environment Implementation
Includes OrderBook class and TradingEnv class
"""

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Tuple, Optional, Dict, Any, List
import math
from dataclasses import dataclass
from collections import defaultdict

# ============================================================================
# Order Book Implementation (Day 1)
# ============================================================================

@dataclass
class Order:
    """Represents a single order in the order book."""
    price: float
    volume: int
    
    def __lt__(self, other: 'Order') -> bool:
        """For sorting: bids sorted descending, asks sorted ascending."""
        return self.price < other.price

class OrderBook:
    """
    A simple limit order book implementation.
    
    Features:
    - Bids (buy orders): sorted by price (highest first)
    - Asks (sell orders): sorted by price (lowest first)
    - Basic order matching
    - Price-time priority
    """
    
    def __init__(self, seed: Optional[int] = None):
        """
        Initialize an empty order book.
        
        Args:
            seed: Random seed for reproducibility
        """
        self.bids: List[Order] = []  # Buy orders (highest price first)
        self.asks: List[Order] = []  # Sell orders (lowest price first)
        self.np_random = np.random.default_rng(seed)
        
    def add_order(self, side: str, price: float, volume: int) -> None:
        """
        Add a new order to the order book.
        
        Args:
            side: 'bid' for buy order, 'ask' for sell order
            price: Order price
            volume: Order volume
        """
        new_order = Order(price, volume)
        
        if side == 'bid':
            # Insert bid in descending price order
            self.bids.append(new_order)
            self.bids.sort(key=lambda x: x.price, reverse=True)
        elif side == 'ask':
            # Insert ask in ascending price order
            self.asks.append(new_order)
            self.asks.sort(key=lambda x: x.price)
        else:
            raise ValueError(f"Invalid side: {side}. Must be 'bid' or 'ask'.")
    
    def get_best_bid(self) -> Optional[Order]:
        """Get the highest bid (best buy price)."""
        return self.bids[0] if self.bids else None
    
    def get_best_ask(self) -> Optional[Order]:
        """Get the lowest ask (best sell price)."""
        return self.asks[0] if self.asks else None
    
    def get_bids(self, levels: int = 5) -> List[Order]:
        """
        Get top N bids.
        
        Args:
            levels: Number of price levels to return
        
        Returns:
            List of top N bids (highest prices first)
        """
        return self.bids[:levels]
    
    def get_asks(self, levels: int = 5) -> List[Order]:
        """
        Get top N asks.
        
        Args:
            levels: Number of price levels to return
        
        Returns:
            List of top N asks (lowest prices first)
        """
        return self.asks[:levels]
    
    def execute_order(self, side: str, price: float, volume: int) -> int:
        """
        Execute an order against the order book.
        
        Args:
            side: 'bid' to execute against asks, 'ask' to execute against bids
            price: Limit price (order only executes at this price or better)
            volume: Volume to execute
        
        Returns:
            Volume actually executed
        """
        executed_volume = 0
        
        if side == 'bid':
            # Buying: execute against asks (sell orders)
            remaining_volume = volume
            asks_to_remove = []
            
            for i, ask in enumerate(self.asks):
                if ask.price <= price and remaining_volume > 0:
                    # Can execute at this price
                    execution_volume = min(ask.volume, remaining_volume)
                    executed_volume += execution_volume
                    
                    # Update ask volume
                    ask.volume -= execution_volume
                    remaining_volume -= execution_volume
                    
                    # Mark empty asks for removal
                    if ask.volume == 0:
                        asks_to_remove.append(i)
                
                elif ask.price > price:
                    # Price too high, stop execution
                    break
            
            # Remove empty asks
            for idx in reversed(asks_to_remove):
                self.asks.pop(idx)
                
        elif side == 'ask':
            # Selling: execute against bids (buy orders)
            remaining_volume = volume
            bids_to_remove = []
            
            for i, bid in enumerate(self.bids):
                if bid.price >= price and remaining_volume > 0:
                    # Can execute at this price
                    execution_volume = min(bid.volume, remaining_volume)
                    executed_volume += execution_volume
                    
                    # Update bid volume
                    bid.volume -= execution_volume
                    remaining_volume -= execution_volume
                    
                    # Mark empty bids for removal
                    if bid.volume == 0:
                        bids_to_remove.append(i)
                
                elif bid.price < price:
                    # Price too low, stop execution
                    break
            
            # Remove empty bids
            for idx in reversed(bids_to_remove):
                self.bids.pop(idx)
        
        return executed_volume
    
    def cancel_order(self, side: str, price: float) -> bool:
        """
        Cancel an order at a specific price.
        
        Args:
            side: 'bid' or 'ask'
            price: Price level to cancel
        
        Returns:
            True if order was found and cancelled
        """
        if side == 'bid':
            for i, bid in enumerate(self.bids):
                if abs(bid.price - price) < 1e-9:  # Float comparison tolerance
                    self.bids.pop(i)
                    return True
        elif side == 'ask':
            for i, ask in enumerate(self.asks):
                if abs(ask.price - price) < 1e-9:
                    self.asks.pop(i)
                    return True
        
        return False
    
    def adjust_prices(self, percentage_change: float) -> None:
        """
        Adjust all prices by a percentage.
        
        Args:
            percentage_change: Percentage change (0.01 = +1%)
        """
        # Adjust bids
        for bid in self.bids:
            bid.price *= (1 + percentage_change)
        
        # Adjust asks
        for ask in self.asks:
            ask.price *= (1 + percentage_change)
        
        # Re-sort after price adjustment
        self.bids.sort(key=lambda x: x.price, reverse=True)
        self.asks.sort(key=lambda x: x.price)
    
    def clear(self) -> None:
        """Clear all orders from the book."""
        self.bids.clear()
        self.asks.clear()
    
    def get_depth(self, levels: int = 5) -> Dict[str, List[Tuple[float, int]]]:
        """
        Get order book depth.
        
        Args:
            levels: Number of price levels to return
        
        Returns:
            Dictionary with 'bids' and 'asks' lists of (price, volume) tuples
        """
        return {
            'bids': [(bid.price, bid.volume) for bid in self.get_bids(levels)],
            'asks': [(ask.price, ask.volume) for ask in self.get_asks(levels)]
        }

# ============================================================================
# Trading Environment Implementation (Day 2)
# ============================================================================

class TradingEnv(gym.Env):
    """
    A deterministic limit order book trading environment for Gymnasium.
    
    This environment implements the complete MDP formulation from Day 1:
    - State: Normalized order book snapshot + portfolio state
    - Actions: Discrete trading decisions
    - Reward: PnL change with transaction costs
    - Transitions: Order book updates based on agent actions
    
    Environment design principles:
    1. Deterministic for debugging
    2. Numerically stable with proper normalization
    3. Compliant with Stable-Baselines3 requirements
    """
    
    def __init__(
        self,
        max_inventory: int = 100,
        initial_cash: float = 10000.0,
        max_steps: int = 1000,
        num_levels: int = 5,
        spread_penalty_weight: float = 0.01,
        inventory_risk_weight: float = 0.001,
        seed: Optional[int] = None
    ):
        """
        Initialize the trading environment.
        
        Args:
            max_inventory: Maximum allowable inventory (positive or negative)
            initial_cash: Starting cash amount
            max_steps: Maximum steps per episode before truncation
            num_levels: Number of price levels to observe in order book
            spread_penalty_weight: Weight for spread penalty in reward
            inventory_risk_weight: Weight for inventory risk penalty
            seed: Random seed for reproducibility
        """
        super().__init__()
        
        # Store parameters
        self.max_inventory = max_inventory
        self.initial_cash = initial_cash
        self.max_steps = max_steps
        self.num_levels = num_levels
        self.spread_penalty_weight = spread_penalty_weight
        self.inventory_risk_weight = inventory_risk_weight
        
        # Set seed for reproducibility
        self._seed = seed
        self.np_random = np.random.default_rng(seed)
        
        # ====================================================================
        # CRITICAL: Observation Space Definition
        # ====================================================================
        # Observation consists of:
        # 1. Order book prices (normalized)
        # 2. Order book volumes (log-normalized)
        # 3. Portfolio state (normalized)
        # 4. Market features (normalized)
        
        # Each price level has: bid_price, bid_volume, ask_price, ask_volume
        # We'll normalize prices relative to mid-price
        # Volumes will be log-scaled
        
        obs_shape = (
            4 * num_levels +  # Order book data (price/volume for bids/asks)
            3 +              # Portfolio: inventory, cash, total_value
            2                # Market: spread, mid_price
        )
        
        # Use float32 for neural network compatibility
        self.observation_space = spaces.Box(
            low=-1.0,  # Negative values allowed for inventory
            high=10.0,  # Higher upper bound for volumes after log scaling
            shape=(obs_shape,),
            dtype=np.float32
        )
        
        # ====================================================================
        # CRITICAL: Action Space Definition
        # ====================================================================
        # Discrete actions:
        # 0: Hold (do nothing)
        # 1: Buy 1 unit at market (execute at best ask)
        # 2: Sell 1 unit at market (execute at best bid)
        # 3: Buy 1 unit with limit order (post bid at spread midpoint)
        # 4: Sell 1 unit with limit order (post ask at spread midpoint)
        
        self.action_space = spaces.Discrete(5)
        
        # ====================================================================
        # State Variables
        # ====================================================================
        self.order_book = None
        self.inventory = 0
        self.cash = initial_cash
        self.current_step = 0
        self.initial_portfolio_value = initial_cash
        self.previous_portfolio_value = initial_cash
        
        # For tracking
        self.total_trades = 0
        self.total_transaction_cost = 0.0
        
        # Normalization parameters (estimated, should be calibrated to your data)
        self.normalization_params = {
            'price_mean': 100.0,      # Estimated mean price
            'price_std': 10.0,        # Estimated price volatility
            'max_volume': 1000000,    # Maximum expected volume
            'min_volume': 1,          # Minimum volume (to avoid log(0))
        }
        
        # For reward calculation tracking
        self.returns_history = []
    
    def _normalize_prices(self, prices: np.ndarray, mid_price: float) -> np.ndarray:
        """
        Normalize prices relative to mid-price.
        
        This is CRITICAL for neural network stability:
        - Prices become relative percentages
        - Avoids large absolute values
        - Preserves relationships between price levels
        
        Formula: (price - mid_price) / mid_price
        """
        if mid_price > 0:
            return (prices - mid_price) / mid_price
        else:
            return np.zeros_like(prices)
    
    def _normalize_volumes(self, volumes: np.ndarray) -> np.ndarray:
        """
        Log-scale volumes to handle extreme ranges.
        
        Why log scaling:
        - Order book volumes often follow power-law distributions
        - Log transform compresses range [1, 1e6] to [0, 6]
        - Makes volumes comparable with normalized prices
        
        Formula: log10(volume + 1) / log10(max_volume + 1)
        """
        max_vol = self.normalization_params['max_volume']
        return np.log10(volumes + 1) / np.log10(max_vol + 1)
    
    def _normalize_inventory(self) -> float:
        """
        Normalize inventory to [-1, 1] range.
        """
        if self.max_inventory > 0:
            return self.inventory / self.max_inventory
        return 0.0
    
    def _normalize_cash(self) -> float:
        """
        Normalize cash to [0, 1] range relative to initial cash.
        """
        return self.cash / (self.initial_cash * 2)  # Allow for growth up to 2x
    
    def _get_market_features(self) -> Tuple[float, float]:
        """
        Extract key market features.
        
        Returns:
            spread: normalized bid-ask spread
            mid_price: current mid-price
        """
        if self.order_book is None:
            return 0.0, 0.0
            
        best_bid = self.order_book.get_best_bid()
        best_ask = self.order_book.get_best_ask()
        
        if best_bid is None or best_ask is None:
            return 0.0, 0.0
        
        mid_price = (best_bid.price + best_ask.price) / 2
        spread = (best_ask.price - best_bid.price) / mid_price if mid_price > 0 else 0.0
        
        return spread, mid_price

    def reset(self, seed: Optional[int] = None, options: Optional[Dict] = None) -> Tuple[np.ndarray, Dict]:
        """
        Reset the environment to initial state.
        
        This is CRITICAL for RL training:
        - Must remove all historical leakage
        - Must provide a valid starting state
        - Must be deterministic when seeded
        
        Returns:
            observation: Initial normalized state
            info: Diagnostic information
        """
        # Handle seeding (Gymnasium requirement)
        if seed is not None:
            self._seed = seed
            self.np_random = np.random.default_rng(seed)
        
        # ====================================================================
        # CRITICAL: Complete State Reset
        # ====================================================================
        
        # Reset order book with deterministic initial state
        self.order_book = OrderBook(seed=self._seed)
        
        # Initialize with realistic order book structure
        self._initialize_order_book()
        
        # Reset portfolio state
        self.inventory = 0
        self.cash = self.initial_cash
        self.current_step = 0
        self.initial_portfolio_value = self.initial_cash
        self.previous_portfolio_value = self.initial_cash
        
        # Reset tracking metrics
        self.total_trades = 0
        self.total_transaction_cost = 0.0
        self.returns_history = []
        
        # Get initial observation
        observation = self._get_observation()
        info = self._get_info()
        
        return observation, info
    
    def _initialize_order_book(self):
        """
        Initialize the order book with realistic starting structure.
        
        This creates a deterministic starting point for debugging.
        """
        # Start with a mid-price around 100
        mid_price = 100.0
        
        # Add bids (below mid-price)
        for i in range(self.num_levels):
            price = mid_price - (i + 1) * 0.5  # 0.5 tick spacing
            volume = self.np_random.integers(100, 1000)
            self.order_book.add_order('bid', price, volume)
        
        # Add asks (above mid-price)
        for i in range(self.num_levels):
            price = mid_price + (i + 1) * 0.5  # 0.5 tick spacing
            volume = self.np_random.integers(100, 1000)
            self.order_book.add_order('ask', price, volume)

    def _get_observation(self) -> np.ndarray:
        """
        Construct the normalized observation vector.
        
        This is the STATE representation (S_t) that the agent observes.
        Normalization is CRITICAL for neural network stability.
        """
        if self.order_book is None:
            return np.zeros(self.observation_space.shape, dtype=np.float32)
        
        # Get raw order book data
        bids = self.order_book.get_bids(self.num_levels)
        asks = self.order_book.get_asks(self.num_levels)
        
        # Get market features for normalization reference
        spread, mid_price = self._get_market_features()
        
        # Initialize observation array
        obs = []
        
        # ====================================================================
        # CRITICAL: Order Book Data Normalization
        # ====================================================================
        
        # Process bids
        for i in range(self.num_levels):
            if i < len(bids):
                bid_price = bids[i].price
                bid_volume = bids[i].volume
            else:
                bid_price = 0.0
                bid_volume = 0.0
            
            # Normalize bid price relative to mid-price
            if mid_price > 0 and bid_price > 0:
                norm_price = (bid_price - mid_price) / mid_price
            else:
                norm_price = -1.0  # Indicate missing level
            
            # Log-normalize volume
            norm_volume = self._normalize_volumes(np.array([bid_volume]))[0]
            
            obs.extend([norm_price, norm_volume])
        
        # Process asks
        for i in range(self.num_levels):
            if i < len(asks):
                ask_price = asks[i].price
                ask_volume = asks[i].volume
            else:
                ask_price = 0.0
                ask_volume = 0.0
            
            # Normalize ask price relative to mid-price
            if mid_price > 0 and ask_price > 0:
                norm_price = (ask_price - mid_price) / mid_price
            else:
                norm_price = 1.0  # Indicate missing level (positive side)
            
            # Log-normalize volume
            norm_volume = self._normalize_volumes(np.array([ask_volume]))[0]
            
            obs.extend([norm_price, norm_volume])
        
        # ====================================================================
        # CRITICAL: Portfolio State Normalization
        # ====================================================================
        
        # Normalize inventory
        norm_inventory = self._normalize_inventory()
        
        # Normalize cash
        norm_cash = self._normalize_cash()
        
        # Calculate and normalize total portfolio value
        total_value = self._calculate_portfolio_value()
        norm_value = total_value / (self.initial_cash * 2)
        
        obs.extend([norm_inventory, norm_cash, norm_value])
        
        # ====================================================================
        # Market Features
        # ====================================================================
        
        # Spread is already normalized (relative spread)
        # Mid-price normalized relative to reference
        norm_mid_price = mid_price / 100.0 if mid_price > 0 else 0.0
        
        obs.extend([spread, norm_mid_price])
        
        # Convert to float32 array (Stable-Baselines3 requirement)
        observation = np.array(obs, dtype=np.float32)
        
        # CRITICAL: Validate no NaNs or Infs
        if not np.isfinite(observation).all():
            # Replace any non-finite values with zeros
            observation = np.nan_to_num(observation, nan=0.0, posinf=0.0, neginf=0.0)
        
        return observation
    
    def _calculate_portfolio_value(self) -> float:
        """
        Calculate total portfolio value (cash + inventory value).
        
        Uses mid-price for inventory valuation (standard accounting practice).
        """
        if self.order_book is None:
            return self.cash
        
        best_bid = self.order_book.get_best_bid()
        best_ask = self.order_book.get_best_ask()
        
        if best_bid is None or best_ask is None:
            mid_price = 100.0  # Default
        else:
            mid_price = (best_bid.price + best_ask.price) / 2
        
        inventory_value = self.inventory * mid_price
        return self.cash + inventory_value

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        """
        Execute one time step in the environment.
        
        This implements the state transition function P(s'|s,a).
        Sequence is CRITICAL:
        1. Validate action
        2. Execute trade
        3. Update market
        4. Calculate reward
        5. Construct observation
        6. Check termination
        
        Args:
            action: Integer action from action_space
        
        Returns:
            observation: Next state
            reward: Immediate reward
            terminated: Episode ended naturally
            truncated: Episode ended artificially
            info: Diagnostic information
        """
        # ====================================================================
        # CRITICAL: Action Validation
        # ====================================================================
        if not self.action_space.contains(action):
            raise ValueError(f"Invalid action: {action}")
        
        # Store previous portfolio value for reward calculation
        previous_value = self._calculate_portfolio_value()
        
        # ====================================================================
        # CRITICAL: Action Execution Logic
        # ====================================================================
        
        # Track transaction costs for this step
        step_transaction_cost = 0.0
        
        if action == 0:  # Hold
            pass  # Do nothing
        
        elif action == 1:  # Buy 1 unit at market
            step_transaction_cost += self._execute_market_buy(quantity=1)
        
        elif action == 2:  # Sell 1 unit at market
            step_transaction_cost += self._execute_market_sell(quantity=1)
        
        elif action == 3:  # Buy with limit order
            step_transaction_cost += self._place_limit_bid()
        
        elif action == 4:  # Sell with limit order
            step_transaction_cost += self._place_limit_ask()
        
        # ====================================================================
        # CRITICAL: Market State Update
        # ====================================================================
        
        # Simulate market activity (other traders)
        self._simulate_market_activity()
        
        # Increment step counter
        self.current_step += 1
        
        # ====================================================================
        # CRITICAL: Reward Calculation
        # ====================================================================
        
        current_value = self._calculate_portfolio_value()
        reward = self._calculate_reward(
            previous_value=previous_value,
            current_value=current_value,
            transaction_cost=step_transaction_cost
        )
        
        # Track returns for statistics
        if previous_value > 0:
            ret = (current_value - previous_value) / previous_value
            self.returns_history.append(ret)
        
        # Update tracking
        self.total_transaction_cost += step_transaction_cost
        if action in [1, 2, 3, 4]:
            self.total_trades += 1
        
        # Update previous value for next step
        self.previous_portfolio_value = current_value
        
        # ====================================================================
        # CRITICAL: Next Observation
        # ====================================================================
        
        observation = self._get_observation()
        
        # ====================================================================
        # CRITICAL: Termination Conditions
        # ====================================================================
        
        terminated = self._check_termination()
        truncated = self.current_step >= self.max_steps
        
        # ====================================================================
        # Diagnostic Information
        # ====================================================================
        
        info = self._get_info()
        info.update({
            'step': self.current_step,
            'action': action,
            'transaction_cost': step_transaction_cost,
            'portfolio_value': current_value,
        })
        
        return observation, reward, terminated, truncated, info

    def _execute_market_buy(self, quantity: int) -> float:
        """
        Execute a market buy order (buy at best ask).
        
        Returns transaction cost incurred.
        """
        if self.order_book is None or quantity <= 0:
            return 0.0
        
        best_ask = self.order_book.get_best_ask()
        if best_ask is None:
            return 0.0
        
        # Calculate cost
        cost = best_ask.price * quantity
        
        # Check if we have enough cash
        if self.cash < cost:
            # Partial execution or reject (simplified: reject)
            return 0.0
        
        # Execute trade
        executed = self.order_book.execute_order('bid', best_ask.price, quantity)
        if executed == 0:
            return 0.0
        
        # Update portfolio
        self.inventory += executed
        self.cash -= best_ask.price * executed
        
        # Calculate transaction cost
        transaction_cost = self._calculate_transaction_cost(
            best_ask.price * executed, 
            is_market_order=True,
            is_buy=True
        )
        
        return transaction_cost
    
    def _execute_market_sell(self, quantity: int) -> float:
        """
        Execute a market sell order (sell at best bid).
        
        Returns transaction cost incurred.
        """
        if self.order_book is None or quantity <= 0:
            return 0.0
        
        # Check if we have enough inventory
        if self.inventory < quantity:
            # Partial execution or reject (simplified: reject)
            return 0.0
        
        best_bid = self.order_book.get_best_bid()
        if best_bid is None:
            return 0.0
        
        # Execute trade
        executed = self.order_book.execute_order('ask', best_bid.price, quantity)
        if executed == 0:
            return 0.0
        
        # Calculate proceeds
        proceeds = best_bid.price * executed
        
        # Update portfolio
        self.inventory -= executed
        self.cash += proceeds
        
        # Calculate transaction cost
        transaction_cost = self._calculate_transaction_cost(
            proceeds,
            is_market_order=True,
            is_buy=False
        )
        
        return transaction_cost
    
    def _place_limit_bid(self) -> float:
        """
        Place a limit bid order at spread midpoint.
        
        Returns expected transaction cost.
        """
        if self.order_book is None:
            return 0.0
        
        best_bid = self.order_book.get_best_bid()
        best_ask = self.order_book.get_best_ask()
        
        if best_bid is None or best_ask is None:
            return 0.0
        
        # Place bid at spread midpoint
        bid_price = (best_bid.price + best_ask.price) / 2
        
        # Check if we have enough cash
        if self.cash < bid_price:
            return 0.0
        
        # Add limit order
        self.order_book.add_order('bid', bid_price, 1)
        
        # Limit orders have lower transaction costs
        transaction_cost = self._calculate_transaction_cost(
            bid_price,
            is_market_order=False,
            is_buy=True
        )
        
        return transaction_cost
    
    def _place_limit_ask(self) -> float:
        """
        Place a limit ask order at spread midpoint.
        
        Returns expected transaction cost.
        """
        if self.order_book is None:
            return 0.0
        
        # Check if we have inventory to sell
        if self.inventory < 1:
            return 0.0
        
        best_bid = self.order_book.get_best_bid()
        best_ask = self.order_book.get_best_ask()
        
        if best_bid is None or best_ask is None:
            return 0.0
        
        # Place ask at spread midpoint
        ask_price = (best_bid.price + best_ask.price) / 2
        
        # Add limit order
        self.order_book.add_order('ask', ask_price, 1)
        
        # Limit orders have lower transaction costs
        transaction_cost = self._calculate_transaction_cost(
            ask_price,
            is_market_order=False,
            is_buy=False
        )
        
        return transaction_cost

    def _calculate_transaction_cost(
        self, 
        value: float, 
        is_market_order: bool,
        is_buy: bool
    ) -> float:
        """
        Calculate transaction costs based on order type and turnover.
        
        CRITICAL: Transaction costs are a major factor in trading profitability.
        This function should reflect realistic market microstructure.
        
        Components:
        1. Fixed commission
        2. Spread cost (for market orders)
        3. High turnover penalty (discourages excessive trading)
        
        Args:
            value: Trade value (price * quantity)
            is_market_order: Whether this is a market order
            is_buy: Whether buying (True) or selling (False)
        
        Returns:
            Total transaction cost in currency units
        """
        # Base commission (0.1% of trade value)
        commission_rate = 0.001
        commission = value * commission_rate
        
        # Spread cost for market orders (0.05% of half-spread)
        spread_cost = 0.0
        if is_market_order and self.order_book:
            best_bid = self.order_book.get_best_bid()
            best_ask = self.order_book.get_best_ask()
            
            if best_bid and best_ask:
                spread = best_ask.price - best_bid.price
                mid_price = (best_bid.price + best_ask.price) / 2
                if mid_price > 0:
                    relative_spread = spread / mid_price
                    # Pay half the spread (conservative estimate)
                    spread_cost = value * relative_spread * 0.5
        
        # High turnover penalty (discourages excessive trading)
        turnover_penalty = 0.0
        if self.current_step > 0:
            # Calculate daily turnover rate
            daily_turnover = self.total_trades / (self.current_step / 100)  # Assuming 100 steps per "day"
            if daily_turnover > 10:  # Penalize turnover > 10x per day
                excess_turnover = daily_turnover - 10
                turnover_penalty = value * excess_turnover * 0.0001
        
        # Directional penalty (optional: can penalize buying more than selling)
        direction_penalty = 0.0
        if is_buy and self.inventory > self.max_inventory * 0.5:
            # Penalize buying when already long
            direction_penalty = value * 0.0005
        
        total_cost = commission + spread_cost + turnover_penalty + direction_penalty
        
        return total_cost

    def _calculate_reward(
        self,
        previous_value: float,
        current_value: float,
        transaction_cost: float
    ) -> float:
        """
        Calculate the reward for the current step.
        
        CRITICAL: The reward function shapes the agent's behavior.
        This is the most important design decision after the state representation.
        
        Reward components:
        1. PnL change (Δportfolio value)
        2. Transaction cost penalty
        3. Inventory risk penalty
        4. Spread penalty (encourages providing liquidity)
        
        Returns:
            Reward scalar (higher is better)
        """
        # 1. PnL change (most important component)
        pnl_change = current_value - previous_value
        
        # 2. Transaction cost penalty (already accounted in pnl, but reinforce)
        cost_penalty = -transaction_cost
        
        # 3. Inventory risk penalty (quadratic in inventory)
        # This encourages mean reversion and prevents large positions
        if self.max_inventory > 0:
            normalized_inventory = abs(self.inventory) / self.max_inventory
            inventory_penalty = -self.inventory_risk_weight * (normalized_inventory ** 2)
        else:
            inventory_penalty = 0.0
        
        # 4. Spread penalty (encourages placing limit orders in wide spreads)
        spread, _ = self._get_market_features()
        spread_penalty = -self.spread_penalty_weight * spread
        
        # 5. Overtrading penalty (if too many trades recently)
        overtrading_penalty = 0.0
        if self.current_step > 10:
            recent_trades = min(10, self.total_trades)
            if recent_trades > 5:
                overtrading_penalty = -0.01 * (recent_trades - 5)
        
        # Total reward
        reward = (
            pnl_change +
            cost_penalty +
            inventory_penalty +
            spread_penalty +
            overtrading_penalty
        )
        
        # CRITICAL: Scale reward for numerical stability
        # Normalize by initial portfolio value
        if self.initial_portfolio_value > 0:
            reward = reward / self.initial_portfolio_value
        
        return float(reward)

    def _simulate_market_activity(self):
        """
        Simulate other traders' activity to create realistic market dynamics.
        
        This creates a non-stationary environment that the agent must adapt to.
        """
        if self.order_book is None:
            return
        
        # Random price movement
        if self.np_random.random() < 0.3:  # 30% chance of price move
            # Get current best prices
            best_bid = self.order_book.get_best_bid()
            best_ask = self.order_book.get_best_ask()
            
            if best_bid and best_ask:
                mid_price = (best_bid.price + best_ask.price) / 2
                
                # Random walk with mean reversion
                price_change = (
                    self.np_random.normal(0, 0.001) +  # Random noise
                    0.01 * (100 - mid_price) / 100  # Mean reversion to 100
                )
                
                # Move all orders by price_change
                self.order_book.adjust_prices(price_change)
        
        # Random order cancellations
        if self.np_random.random() < 0.2:  # 20% chance of cancellation
            # Cancel random order
            if self.np_random.random() < 0.5:
                side = 'bid'
            else:
                side = 'ask'
            
            if side == 'bid' and self.order_book.bids:
                order_to_cancel = self.np_random.choice(self.order_book.bids)
                self.order_book.cancel_order(side, order_to_cancel.price)
            elif side == 'ask' and self.order_book.asks:
                order_to_cancel = self.np_random.choice(self.order_book.asks)
                self.order_book.cancel_order(side, order_to_cancel.price)
        
        # Random new orders
        if self.np_random.random() < 0.4:  # 40% chance of new order
            side = 'bid' if self.np_random.random() < 0.5 else 'ask'
            
            # Place near current prices
            best_bid = self.order_book.get_best_bid()
            best_ask = self.order_book.get_best_ask()
            
            if best_bid and best_ask:
                if side == 'bid':
                    price = best_bid.price - self.np_random.uniform(0, 0.5)
                else:
                    price = best_ask.price + self.np_random.uniform(0, 0.5)
                
                volume = self.np_random.integers(10, 1000)
                self.order_book.add_order(side, price, volume)
    
    def _check_termination(self) -> bool:
        """
        Check if the episode should terminate naturally.
        
        Termination conditions:
        1. Bankruptcy (cash < 0 or portfolio value < threshold)
        2. Inventory limits exceeded
        3. Maximum steps reached (handled by truncation)
        """
        # Bankruptcy check
        portfolio_value = self._calculate_portfolio_value()
        if portfolio_value < self.initial_cash * 0.1:  # Lost 90% of initial value
            return True
        
        # Inventory limit check
        if abs(self.inventory) > self.max_inventory:
            return True
        
        # Cash bankruptcy
        if self.cash < 0:
            return True
        
        return False
    
    def _get_info(self) -> Dict[str, Any]:
        """
        Return diagnostic information for monitoring.
        
        Info dictionary should contain metrics but NOT any information
        that could be used by the agent (that would be cheating).
        """
        portfolio_value = self._calculate_portfolio_value()
        
        # Calculate Sharpe ratio (simplified)
        sharpe = 0.0
        if len(self.returns_history) > 1:
            returns = np.array(self.returns_history)
            sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252)  # Annualized
        
        return {
            'inventory': self.inventory,
            'cash': self.cash,
            'portfolio_value': portfolio_value,
            'total_trades': self.total_trades,
            'total_transaction_cost': self.total_transaction_cost,
            'return': (portfolio_value - self.initial_cash) / self.initial_cash,
            'sharpe_ratio': sharpe,
            'position_pct': self.inventory / self.max_inventory if self.max_inventory > 0 else 0.0,
        }
    
    def render(self, mode: str = 'human'):
        """
        Render the current state of the environment.
        
        Useful for debugging and visualization.
        """
        if mode == 'human':
            print(f"\n=== Step {self.current_step} ===")
            print(f"Inventory: {self.inventory}")
            print(f"Cash: ${self.cash:.2f}")
            
            portfolio_value = self._calculate_portfolio_value()
            print(f"Portfolio Value: ${portfolio_value:.2f}")
            print(f"Return: {(portfolio_value - self.initial_cash)/self.initial_cash*100:.2f}%")
            
            if self.order_book:
                best_bid = self.order_book.get_best_bid()
                best_ask = self.order_book.get_best_ask()
                
                if best_bid and best_ask:
                    print(f"Bid/Ask: {best_bid.price:.2f} / {best_ask.price:.2f}")
                    print(f"Spread: {(best_ask.price - best_bid.price):.2f}")
            
            print(f"Total Trades: {self.total_trades}")
            print(f"Total Transaction Cost: ${self.total_transaction_cost:.2f}")

# ============================================================================
# Testing and Validation
# ============================================================================

def test_environment():
    """
    Comprehensive test of the TradingEnv implementation.
    
    This validates:
    1. Gymnasium API compliance
    2. Numerical stability
    3. Deterministic behavior
    4. Reward consistency
    """
    
    print("Testing Trading Environment...")
    print("=" * 60)
    
    # Test 1: Basic initialization
    print("\n1. Testing initialization...")
    env = TradingEnv(seed=42)
    
    assert env.observation_space is not None
    assert env.action_space is not None
    print("✓ Observation space:", env.observation_space)
    print("✓ Action space:", env.action_space)
    
    # Test 2: Reset functionality
    print("\n2. Testing reset...")
    obs, info = env.reset(seed=123)
    
    assert obs.shape == env.observation_space.shape
    assert obs.dtype == np.float32
    assert np.isfinite(obs).all(), "Observation contains NaN or Inf"
    print(f"✓ Observation shape: {obs.shape}")
    print(f"✓ Observation dtype: {obs.dtype}")
    print(f"✓ All values finite: {np.isfinite(obs).all()}")
    
    # Test 3: Step through a few actions
    print("\n3. Testing step function...")
    
    rewards = []
    for i in range(10):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        
        # Validate returns
        assert obs.shape == env.observation_space.shape
        assert isinstance(reward, float)
        assert isinstance(terminated, bool)
        assert isinstance(truncated, bool)
        assert isinstance(info, dict)
        
        # Check numerical stability
        assert np.isfinite(obs).all(), f"Step {i}: Observation contains NaN or Inf"
        assert np.isfinite(reward), f"Step {i}: Reward is NaN or Inf"
        
        rewards.append(reward)
        
        if terminated or truncated:
            break
    
    print(f"✓ Executed {len(rewards)} steps")
    print(f"✓ Reward range: [{min(rewards):.4f}, {max(rewards):.4f}]")
    
    # Test 4: Deterministic behavior with same seed
    print("\n4. Testing determinism...")
    
    env1 = TradingEnv(seed=42)
    env2 = TradingEnv(seed=42)
    
    obs1, _ = env1.reset()
    obs2, _ = env2.reset()
    
    # First reset should be identical
    assert np.allclose(obs1, obs2), "Different observations with same seed"
    
    # Steps should also be identical
    for _ in range(5):
        action = 0  # Hold action
        obs1, reward1, term1, trunc1, info1 = env1.step(action)
        obs2, reward2, term2, trunc2, info2 = env2.step(action)
        
        assert np.allclose(obs1, obs2), "Observation mismatch"
        assert np.allclose(reward1, reward2), "Reward mismatch"
        assert term1 == term2, "Termination mismatch"
        assert trunc1 == trunc2, "Truncation mismatch"
    
    print("✓ Environment is deterministic with same seed")
    
    # Test 5: Episode termination
    print("\n5. Testing termination conditions...")
    
    # Test bankruptcy
    env_bankrupt = TradingEnv(initial_cash=10.0, seed=42)
    obs, _ = env_bankrupt.reset()
    
    # Force bad trades until bankruptcy
    for i in range(100):
        # Always buy at high prices
        obs, reward, terminated, truncated, info = env_bankrupt.step(1)  # Buy
        
        if terminated:
            print(f"✓ Episode terminated at step {i} (bankruptcy)")
            break
    
    # Test 6: Observation normalization
    print("\n6. Testing observation normalization...")
    
    env_norm = TradingEnv(seed=42)
    obs, _ = env_norm.reset()
    
    # Check that values are in reasonable ranges
    assert (obs >= env_norm.observation_space.low).all(), "Values below lower bound"
    assert (obs <= env_norm.observation_space.high).all(), "Values above upper bound"
    
    # Check that portfolio-related values are normalized
    portfolio_idx = 4 * env_norm.num_levels * 2  # Start of portfolio section
    portfolio_values = obs[portfolio_idx:portfolio_idx+3]
    
    print(f"✓ Portfolio values (norm inventory, cash, value): {portfolio_values}")
    print("✓ All observations within defined bounds")
    
    print("\n" + "=" * 60)
    print("All tests passed! Environment is ready for RL training.")
    
    return env

# ============================================================================
# Main Execution
# ============================================================================

if __name__ == "__main__":
    # Run tests
    env = test_environment()
    
    # Quick demonstration
    print("\n\nEnvironment Demonstration:")
    print("-" * 40)
    
    obs, info = env.reset()
    print(f"Initial observation shape: {obs.shape}")
    print(f"Initial portfolio value: ${info['portfolio_value']:.2f}")
    print(f"Observation sample (first 10 values): {obs[:10]}")
    
    # Run a few steps with rendering
    print("\nRunning 5 steps with random actions:")
    for i in range(5):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        
        env.render()
        print(f"Step {i+1}: Action={action}, Reward={reward:.4f}")
        print(f"Portfolio: ${info['portfolio_value']:.2f}, Inventory: {info['inventory']}")
        print("-" * 30)
        
        if terminated or truncated:
            print("Episode ended!")
            break
    
    print("\n✓ Environment is working correctly!")

Testing Trading Environment...

1. Testing initialization...
✓ Observation space: Box(-1.0, 10.0, (25,), float32)
✓ Action space: Discrete(5)

2. Testing reset...
✓ Observation shape: (25,)
✓ Observation dtype: float32
✓ All values finite: True

3. Testing step function...
✓ Executed 10 steps
✓ Reward range: [-0.0002, -0.0000]

4. Testing determinism...
✓ Environment is deterministic with same seed

5. Testing termination conditions...

6. Testing observation normalization...
✓ Portfolio values (norm inventory, cash, value): []
✓ All observations within defined bounds

All tests passed! Environment is ready for RL training.


Environment Demonstration:
----------------------------------------
Initial observation shape: (25,)
Initial portfolio value: $10000.00
Observation sample (first 10 values): [-0.005       0.4949026  -0.01        0.4016555  -0.015       0.4749032
 -0.02        0.43034387 -0.025       0.4493927 ]

Running 5 steps with random actions:

=== Step 1 ===
Inventory: 0
Cas